# Base de Datos
### De los proyectos pasados se construye una base de datos con las características extraídas en cada proyecto


In [1]:
import pandas as pd

# Primero cargamos los archivos que contienen las características de los proyectos anteriores
df_hrv = pd.read_excel("./Data/DatosFinalNk2.xlsx")
df_dct = pd.read_excel("./Data/DatosConDCT.xlsx")
df_dft = pd.read_excel("./Data/DatosConDFT.xlsx")
df_mfcc = pd.read_excel("./Data/DatosConMfcc1.xlsx")

# Visualizar las primeras filas de cada dataframe para analizarlos
df_hrv.head(), df_dct.head(), df_dft.head(), df_mfcc.head()


(                     FileName Rhythm      Beat  PatientAge Gender  \
 0  MUSE_20180712_160452_75000   AFIB       LVH          74   MALE   
 1  MUSE_20180118_131905_87000   AFIB       TWC          72   MALE   
 2  MUSE_20180114_071444_05000   AFIB       TWC          74   MALE   
 3  MUSE_20180114_132956_24000   AFIB  LVHV TWC          68   MALE   
 4  MUSE_20180118_124852_18000   AFIB      STTU          78   MALE   
 
    VentricularRate  AtrialRate  QRSDuration  QTInterval  QTCorrected  ...  \
 0               49          57           92         488          440  ...   
 1              151         170           84         298          472  ...   
 2              107         108           82         342          456  ...   
 3               95          87           84         334          419  ...   
 4               97         153          104         344          436  ...   
 
    QRSCount  QOnset  QOffset  TOffset  AgeGroup        SDNN       RMSSD  \
 0         7     221      267   

Seguidamente hacemos un merge usando la columna FileName como clave, y construiremos un dataframe con las siguientes columnas identificadores: FileName, Rhythm

Proyecto 1: SDNN y RMSSD

Proyecto 2 – DFT: frecuencia_pico, energia_total

Proyecto 2 – DCT: dct_mean, dct_std

Proyecto 2 – MFCC: mfcc_energy, mfcc_entropy

In [2]:
# Seleccionamos las columnas clave de cada dataframe
df_hrv_sel = df_hrv[['FileName', 'Rhythm', 'SDNN', 'RMSSD']]
df_dct_sel = df_dct[['FileName', 'dct_mean', 'dct_std']]
df_dft_sel = df_dft[['FileName', 'frecuencia_pico', 'energia_total']]
df_mfcc_sel = df_mfcc[['FileName', 'mfcc_energy', 'mfcc_entropy']]

# Unimos todos los dataframes usando 'FileName' como clave
df_merge_1 = pd.merge(df_hrv_sel, df_dct_sel, on='FileName', how='inner')
df_merge_2 = pd.merge(df_merge_1, df_dft_sel, on='FileName', how='inner')
df_final = pd.merge(df_merge_2, df_mfcc_sel, on='FileName', how='inner')

# Mostrar el dataframe final
df_final.to_excel("Data/DataframeTrabajoFinal.xlsx", index=False)



### Características adicionales (Punto 1 del presente proyecto – Artículo de Zheng et al.):
Sólo utilizaremos 2 de las características que se hablan en el artículo, las cuales son: 

* La prominencia del pico más alto (probable QRS): prominencia_QRS: la prominencia del pico más alto (probable QRS).
* El número de picos pequeños que no son QRS: num_peaks_no_QRS: cantidad de picos pequeños que no son QRS (con prominencia < 40% del máximo)

In [3]:
import pandas as pd
import os
from scipy.signal import find_peaks, peak_prominences

# Cargamos el dataframe principal
df_final = pd.read_excel("./Data/DataframeTrabajoFinal.xlsx")

df_final["FileName"] = df_final["FileName"].astype(str)
df_final["FileName"] = df_final["FileName"].apply(lambda x: x if x.endswith(".csv") else x + ".csv")

# Ruta de archivos ECG filtrados
carpeta_ecg = "./Data/ECGDataDenoised"
archivos = [f for f in os.listdir(carpeta_ecg) if f.endswith('.csv')]

# Calculamos características morfológicas
nombres = []
prominencias = []
picos_no_qrs = []


In [4]:
for archivo in archivos:
    try:
        ruta = os.path.join(carpeta_ecg, archivo)
        df = pd.read_csv(ruta, header=None)

        # Seleccionamos la derivación II (columna 1)
        derivacion_ii = df[1].values

        peaks, _ = find_peaks(derivacion_ii, distance=150)
        prominences_calc = peak_prominences(derivacion_ii, peaks)[0]

        prom_max = prominences_calc.max() if len(prominences_calc) > 0 else 0
        umbral = 0.4 * prom_max
        no_qrs_count = sum(prominences_calc < umbral)

        nombres.append(archivo)
        prominencias.append(prom_max)
        picos_no_qrs.append(no_qrs_count)

    except Exception as e:
        nombres.append(archivo)
        prominencias.append(None)
        picos_no_qrs.append(None)

# Creamos el DataFrame con nuevas características
df_morf = pd.DataFrame({
    "FileName": nombres,
    "prominencia_QRS": prominencias,
    "num_peaks_no_QRS": picos_no_qrs
})

# Unimos con el dataframe principal
df_actualizado = pd.merge(df_final, df_morf, on="FileName", how="left")

# Guardamos el archivo final actualizado
df_actualizado.to_excel("./Data/DataframeTrabajoFinal.xlsx", index=False)


### Ahora, eliminaremos del Dataframe las arritmias no seleccionadas. Dejaremos sólo SR, AFIB, SB y SVT:

* ***Ritmo Sinusal (SR):*** Corresponde al ritmo cardíaco normal, generado de forma regular por el nodo sinoauricular. Se elige ya que puede ser la ritmia de referencia para marcar diferencias con otras arritmias, ya que su morfología típica (onda P seguida de complejo QRS y onda T, con intervalos regulares) permite contrastar con ritmos patológicos y calibrar la sensibilidad del sistema frente a desviaciones.


* ***Fibrilación Auricular (AFIB):*** Se elige porque es una de las arritmias más comunes y clínicamente importantes. Se caracteriza por la ausencia de onda P visible [6], presencia de múltiples ondas de fibrilación y una variabilidad marcada en los intervalos RR con un latido rápido y descoordinado. Representa un caso en el que se pierden componentes clásicos del ECG, lo que desafía tanto la extracción de características como la capacidad de generalización de los clasificadores.


* ***Bradicardia Sinusal (SB):*** Se elige porque representa un ritmo de origen sinusal pero con frecuencia cardíaca disminuida (generalmente < 60 latidos por minuto). En la bradicardia sinusal el ECG muestra ondas P seguidas de QRS de apariencia normal, con un patrón regular pero a menor frecuencia. Aunque mantiene la morfología típica del ritmo sinusal, los intervalos entre latidos son más largos, lo cual es útil para entrenar modelos que detecten cambios en frecuencia sin alteraciones en la forma de onda [3].


* ***Taquicardia Supraventricular (SVT):*** Se elige porque agrupa es una arritmia rápida, la cual puede contrastar con la bradicardia. Su detección es compleja, ya que muchas veces la onda P está superpuesta con el complejo QRS o ausente visualmente, y el ritmo tiende a ser muy regular, lo cual nos permitirá evaluar si el modelo es capaz de reconocer ritmos rápidos y regulares con morfologías menos evidentes.


In [ ]:
# Cargamos el DataFrame creado
df = pd.read_excel("./Data/DataframeTrabajoFinal.xlsx")

# Filtramos solo las arritmias seleccionadas
arritmias_seleccionadas = ['SR', 'AFIB', 'SB', 'SVT']
df_filtrado = df[df['Rhythm'].isin(arritmias_seleccionadas)].copy()

# Guardamos sobre el mismo archivo
ruta_guardado = "./Data/DataframeTrabajoFinal.xlsx"
df_filtrado.to_excel(ruta_guardado, index=False)

ruta_guardado

'./Data/DataframeTrabajoFinal.xlsx'